# Phase 6 v2 — GQA base, LoRMA+, majority-coverage long-text summarization
**Run on Kaggle GPU T4 ×2**, Internet enabled, with the `phase2v2` dataset attached. Use **Save Version → Save & Run All**. This is a complete notebook; run cells in order.

## Suggested differences from v3, implemented here
- **LoRMA+ only:** identity-inflated multiplicative adaptation instead of additive LoRA. No student, knowledge distillation or adapter ablation. The saved runtime goes to this one model.
- **Exact v2 base:** unchanged architecture/tokenizer source from `phase-2-v2 (7).ipynb`. Strictly load `/kaggle/input/datasets/koooshiieee/phase2v2/model.pt`, keeping its GQA and its checkpoint-defined RoPE/ALiBi. Do not substitute v3 MLA weights or a Phase 2 NER head.
- **Read a measured majority of every document:** overlapping windows, legally salient windows mixed with windows spread across the judgment, always retaining opening/outcome windows. Try at least **75% unique source-token coverage per document**, then 65% or 55% if measured GPU throughput requires it. This never means dropping test documents; every test document receives a prediction. Longer documents may need many windows to retain a majority.
- **Train on relevant information:** lexical reference-sentence-to-source-chunk alignment for training only, rather than asking each chunk to predict the complete reference. No reference-based test retrieval or external RAG. Targets cropped to the length cap are not given a false EOS.
- **Budget-conscious outputs:** 640-token final document summary, adaptive per-window decoding, cached/fused attention, two-GPU batching, and generated-sentence deduplication. No untrained generative rewrite pass.

Reasonable later experiments, one at a time: rank 16 → 32, 75% → 90% coverage with a larger budget, learning rate 2e-4 → 1e-4, or 640 → 768 final-summary tokens. These are suggestions, not additional runs hidden in this one-hour notebook. Defaults below run one LoRMA+ model.

**LoRMA definition:** `W_eff = (I + (alpha/r) B A) W`; low-rank matrices act on `W x`, not on `x`. This is the additive-identity-inflation variant LoRMA+, not permutation-inflation LoRMAπ. Base linear weights/embeddings are frozen; norms also adapt. The multiplicative branch starts at zero, preserving base outputs initially. See the [LoRMA paper](https://aclanthology.org/2025.findings-acl.527/).

**One-hour target:** the clock starts in settings before dependency installation. A 54-minute internal budget and 59-minute subprocess deadline leave room before 60 minutes. Source coverage adapts to a measured generation benchmark, and training stops early enough to reserve inference/packaging, with extra headroom for the verified HF upload. If a majority-coverage complete run cannot fit, it fails clearly instead of silently emitting incomplete predictions. Downloads, stalls and actual Kaggle throughput prevent an unconditional runtime guarantee. The genuine-checkpoint GPU run has not been performed in this authoring environment.

**Final output only:** `/kaggle/working/phase6_v2_lorma_predictions/submission_lorma/prediction.json` and `submission_lorma.zip`, whose sole root member is `prediction.json`. Best trainable parameters stay in CPU memory during the run. No model weight file is written or exported. Temporary token data/shards are removed after the ZIP verifies. Metrics remain in notebook output.


**Corrected Hugging Face export:** enable Kaggle secret `HF_WRITE` (or environment `HF_WRITE_TOKEN`) with write access to `DumbLad/legal-phase6-v2`. The preflight checks identity, repository creation/access and write permission **before expensive preparation/training**. A fine-grained token must cover this repository. No secret is written into CONFIG or printed. This notebook uploads only the two verified prediction artifacts under `phase6_v2/`, then downloads them at the returned commit and compares SHA-256 hashes. It does not upload weights, caches, notebooks or all of `/kaggle/working`. Existing unrelated remote files are not deleted. Both credential checks and upload run inside the 59-minute global subprocess deadline. Local predictions remain available if the network upload fails.

The original attached notebook had reintroduced adapter/merged weight saves and disabled cleanup while still claiming no weights were saved. This copy restores the earlier prediction-only requirement. No actual Hub upload or full Kaggle run has been performed during this local review. API usage follows the [HF upload guide](https://huggingface.co/docs/huggingface_hub/guides/upload) and [HfApi auth_check documentation](https://huggingface.co/docs/huggingface_hub/package_reference/hf_api#huggingface_hub.HfApi.auth_check).


In [ ]:
# Phase 6 v2 settings and one-hour clock
import os, sys, json, time, tempfile
from pathlib import Path
RUN_STARTED_AT = time.time()
CONFIG = dict(
    hf_repo_id="DumbLad/legal-phase6-v2", hf_path_in_repo="phase6_v2",
    seed=67, checkpoint="/kaggle/input/datasets/koooshiieee/phase2v2/model.pt",
    dataset_repo="Exploration-Lab/CS-781-Capstone",
    train_remote="Phase_6_Summ/Train/train.json", test_remote="Phase_6_Summ/Test/test.json",
    tokenizer_repo="DumbLad/legal-bpe-tokenizer-phase1-v2",
    runtime=str(Path(tempfile.gettempdir())/"phase6_v2_lorma_runtime"),
    tokenizer_dir=str(Path(tempfile.gettempdir())/"phase6_v2_lorma_runtime"/"tokenizer"),
    submission_dir="/kaggle/working/phase6_v2_lorma_predictions",
    input_tokens=640, target_tokens=384, chunk_overlap=128,
    train_coverage=0.70, inference_coverage_options=[0.75,0.65,0.55],
    alignment_min_similarity=0.08,
    summary_tokens=640, min_chunk_new_tokens=48, max_chunk_new_tokens=160,
    rank=16, alpha=16, dropout=0.05, lr=2e-4,
    micro_batch=2, grad_accum=2, gen_batch=8,
    warmup_steps=40, max_steps=2400, min_updates=40, eval_every=100,
    val_loss_docs=32, val_loss_chunks=32, val_gen_docs=8,
    minimum_training_minutes=18, budget_minutes=54, started_at=RUN_STARTED_AT,
)
assert Path(CONFIG['checkpoint']).is_file(), f"Attach phase2v2; missing {CONFIG['checkpoint']}"
Path(CONFIG['runtime']).mkdir(parents=True,exist_ok=True)
Path('phase6_v2_config.json').write_text(json.dumps(CONFIG,indent=2))


In [ ]:
# Install small dependencies, require two GPUs and define the bounded runner
import torch, subprocess, signal, threading
assert torch.cuda.is_available() and torch.cuda.device_count()==2, "Choose GPU T4 x2 in Kaggle settings."
print('PyTorch:',torch.__version__)
for gpu in range(2):print('GPU',gpu,torch.cuda.get_device_name(gpu))
if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF')
    except Exception:print('No optional HF secret; attempting public downloads.')

def run_checked(args):
    remaining=RUN_STARTED_AT+59*60-time.time()
    assert remaining>0,'59-minute subprocess deadline reached.'
    process=subprocess.Popen(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
    def stream():
        for line in process.stdout:print(line,end='',flush=True)
    thread=threading.Thread(target=stream,daemon=True);thread.start()
    try:status=process.wait(timeout=remaining)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid,signal.SIGTERM)
        try:process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid,signal.SIGKILL);process.wait()
        thread.join(timeout=5)
        raise RuntimeError('Execution exceeded its 59-minute deadline. No completed submission is claimed.')
    thread.join(timeout=10)
    assert status==0,f'Run failed with exit code {status}; inspect the error above.'

# Dependency downloads are covered by the same global deadline.
run_checked([sys.executable,'-m','pip','install','-q','huggingface_hub','regex','rouge-score','tqdm','--retries','1','--timeout','30'])


Exact unchanged source copy from the Phase 2 v2 cell beginning `%%writefile llama_scratch.py`. The original pretrained LM architecture is preserved.

In [ ]:
%%writefile llama_scratch.py
"""
Llama-style decoder-only LM, written from scratch (no HF / no llama library).

Pieces: RMSNorm, RoPE (or ALiBi), grouped-query attention (GQA; MQA = n_kv_heads 1,
MHA = n_kv_heads == n_heads), SwiGLU MLP, pre-norm residual blocks, tied embeddings,
KV-cache generation with left-padded batches.

Only torch tensors/autograd/nn.Linear/nn.Embedding are used as building blocks.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------- config
@dataclass
class Config:
    vocab_size: int = 33000
    d_model: int = 1024
    n_layers: int = 16
    n_heads: int = 16
    n_kv_heads: int = 4          # 16 query heads share 4 KV heads (4 queries per KV head)
    ffn_hidden: int = 2560
    max_seq_len: int = 1024      # training context
    rope_cache_len: int = 4096   # RoPE table length (>= prompt + generated tokens)
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    tie_embeddings: bool = True
    pos_enc: str = "rope"        # "rope" | "alibi"
    attn_impl: str = "manual"    # "manual" (hand-written softmax(QK^T)V) | "sdpa" (torch fused kernel, same math)


PRESETS = {
    # exact totals with vocab 33000, tied embeddings
    "v2_100m": dict(d_model=768, n_layers=12, n_heads=12, n_kv_heads=4, ffn_hidden=2048), # ~100.86M
    "target_150m": dict(d_model=896, n_layers=14, n_heads=14, n_kv_heads=2, ffn_hidden=2304), # ~142M 
    "base":  dict(d_model=1024, n_layers=16, n_heads=16, n_kv_heads=4, ffn_hidden=2560),  # ~201.6M
    "small": dict(d_model=1024, n_layers=12, n_heads=16, n_kv_heads=4, ffn_hidden=2816),  # ~169.1M
    "tiny":  dict(d_model=64,   n_layers=2,  n_heads=4,  n_kv_heads=2, ffn_hidden=128),   # smoke tests
}


def analytic_param_count(c: Config) -> int:
    """Closed-form parameter count (use it to size a model before building it)."""
    hd = c.d_model // c.n_heads
    attn = 2 * c.d_model * c.n_heads * hd + 2 * c.d_model * c.n_kv_heads * hd   # Wq,Wo + Wk,Wv
    mlp = 3 * c.d_model * c.ffn_hidden                                           # gate, up, down
    per_layer = attn + mlp + 2 * c.d_model                                       # + 2 RMSNorm
    total = per_layer * c.n_layers + c.vocab_size * c.d_model + c.d_model        # + embedding + final norm
    if not c.tie_embeddings:
        total += c.vocab_size * c.d_model
    return total


def count_params(model, verbose=True) -> int:
    """model.parameters() yields a tied (shared) tensor only once, so tied weights are counted once."""
    total = sum(p.numel() for p in model.parameters())
    if verbose:
        emb = model.tok_emb.weight.numel()
        blocks = sum(p.numel() for p in model.layers.parameters())
        print(f"total {total/1e6:.1f}M | transformer blocks {blocks/1e6:.1f}M | embedding {emb/1e6:.1f}M "
              f"({'tied with output head' if model.cfg.tie_embeddings else 'output head separate'}) "
              f"| analytic {analytic_param_count(model.cfg)/1e6:.1f}M")
    return total


# ----------------------------------------------------------------------------- RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()                                   # always normalise in fp32
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)


# ----------------------------------------------------------------------------- RoPE
def build_rope_cache(head_dim, max_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))   # (hd/2,)
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)                                                # (max_len, hd/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x: (B, H, T, hd); cos/sin: (T, hd/2). Rotates the two halves of each head vector (Llama layout)."""
    half = x.shape[-1] // 2
    xf = x.float()
    x1, x2 = xf[..., :half], xf[..., half:]
    cos = cos[None, None]
    sin = sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


# ----------------------------------------------------------------------------- ALiBi
def get_alibi_slopes(n_heads):
    def pow2(n):
        start = 2 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_heads).is_integer():
        s = pow2(n_heads)
    else:
        c = 2 ** math.floor(math.log2(n_heads))
        s = pow2(c) + pow2(2 * c)[0::2][: n_heads - c]
    return torch.tensor(s, dtype=torch.float32)


# ----------------------------------------------------------------------------- GQA
def repeat_kv(x, n_rep):
    """(B, n_kv, S, hd) -> (B, n_kv*n_rep, S, hd); query head h uses kv head h // n_rep."""
    if n_rep == 1:
        return x
    B, n_kv, S, hd = x.shape
    return x[:, :, None].expand(B, n_kv, n_rep, S, hd).reshape(B, n_kv * n_rep, S, hd)


class GQAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.nh, self.nkv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.n_rep = self.nh // self.nkv
        self.impl = cfg.attn_impl
        self.wq = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)    # (B, nh,  T, hd)
        k = self.wk(x).view(B, T, self.nkv, self.hd).transpose(1, 2)   # (B, nkv, T, hd)
        v = self.wv(x).view(B, T, self.nkv, self.hd).transpose(1, 2)

        if cos is not None:                                            # RoPE on q and k (not v)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past is not None:                                           # cache holds only nkv heads
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        new_cache = (k, v) if use_cache else None
        S = k.shape[2]

        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # relative distance of every (query, key) pair; keys j <= query position are visible
        qpos = torch.arange(S - T, S, device=x.device)
        kpos = torch.arange(S, device=x.device)
        rel = qpos[:, None] - kpos[None, :]                            # (T, S)
        allowed = (rel >= 0)[None, None]                               # (1, 1, T, S)
        if key_valid is not None:                                      # hide left-padding keys
            allowed = allowed & key_valid[:, None, None, :]
        bias = None
        if alibi_slopes is not None:
            bias = -alibi_slopes.view(1, -1, 1, 1) * rel.clamp(min=0).float()[None, None]   # (1, nh, T, S)

        if self.impl == "sdpa" and key_valid is None and T == S:      # training fast path
            if bias is None:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            else:
                m = bias.masked_fill(~allowed, torch.finfo(q.dtype).min).to(q.dtype)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        else:                                                          # hand-written attention
            scores = ((q * (self.hd ** -0.5)) @ k.transpose(-2, -1)).float()   # (B, nh, T, S)
            if bias is not None:
                scores = scores + bias
            scores = scores.masked_fill(~allowed, -1e30)               # finite, so a fully-masked pad row can't give NaN
            probs = torch.softmax(scores, dim=-1).to(v.dtype)
            out = probs @ v                                            # (B, nh, T, hd)

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.wo(out), new_cache


# ----------------------------------------------------------------------------- SwiGLU
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden, bias=False)
        self.w_up = nn.Linear(d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


# ----------------------------------------------------------------------------- block + model
class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg.d_model, cfg.ffn_hidden)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        h, new_cache = self.attn(self.attn_norm(x), cos, sin, alibi_slopes, key_valid, past, use_cache)
        x = x + h
        x = x + self.mlp(self.mlp_norm(x))
        return x, new_cache


class LlamaLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.pos_enc in ("rope", "alibi") and cfg.attn_impl in ("manual", "sdpa")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        if cfg.pos_enc == "rope":
            cos, sin = build_rope_cache(cfg.d_model // cfg.n_heads, cfg.rope_cache_len, cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
            self.alibi_slopes = None
        else:
            self.rope_cos = self.rope_sin = None
            self.register_buffer("alibi_slopes", get_alibi_slopes(cfg.n_heads), persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():       # GPT-2 style: shrink the residual-branch output projections
            if name.endswith("wo.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False, cache=None, start_pos=0,
                key_valid=None, all_logits=False):
        """
        training:  model(x, targets=y) -> scalar loss
        inference: model(idx, use_cache=True, cache=..., start_pos=..., key_valid=...) -> (logits, cache)
                   logits are for the last position only unless all_logits=True.
        """
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos = sin = None
        if self.cfg.pos_enc == "rope":
            assert start_pos + T <= self.cfg.rope_cache_len, "sequence longer than the RoPE table"
            cos = self.rope_cos[start_pos:start_pos + T]
            sin = self.rope_sin[start_pos:start_pos + T]

        new_caches = []
        for i, layer in enumerate(self.layers):
            past = cache[i] if cache is not None else None
            x, c = layer(x, cos, sin, self.alibi_slopes, key_valid, past, use_cache)
            new_caches.append(c)
        x = self.norm(x)

        if targets is not None:
            logits = self.lm_head(x)
            return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        logits = self.lm_head(x if all_logits else x[:, -1:, :])
        return logits, (new_caches if use_cache else None)


# ----------------------------------------------------------------------------- generation
@torch.no_grad()
def generate(model, prompts, max_new_tokens, eos_id, pad_id, temperature=0.1, top_k=30,
             top_p=0.95, rep_penalty=1.2, ban_eos=True):
    """
    prompts: list of lists of token ids (any lengths). Left-pads them into one batch, runs one prefill,
    then decodes with a KV cache. Returns a list of lists of generated ids (EOS / padding removed).
    temperature <= 0 -> greedy.
    """
    model.eval()
    device = next(model.parameters()).device
    V = model.cfg.vocab_size
    B = len(prompts)
    L = max(len(p) for p in prompts)
    ids = torch.full((B, L), pad_id, dtype=torch.long, device=device)
    key_valid = torch.zeros(B, L, dtype=torch.bool, device=device)
    seen = torch.zeros(B, V, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        t = torch.tensor(p, dtype=torch.long, device=device)
        ids[b, L - len(p):] = t
        key_valid[b, L - len(p):] = True
        seen[b, t] = True

    out = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    amp = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda"))
    with amp:
        logits, cache = model(ids, use_cache=True, key_valid=key_valid, start_pos=0)
        for step in range(max_new_tokens):
            lg = logits[:, -1, :].float()
            if ban_eos:
                lg[:, eos_id] = float("-inf")
            if rep_penalty != 1.0:                                     # CTRL-style repetition penalty
                pen = torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty)
                lg = torch.where(seen, pen, lg)
            if temperature <= 0:
                nxt = lg.argmax(-1)
            else:
                lg = lg / temperature
                if top_k and top_k > 0:
                    kth = torch.topk(lg, min(top_k, V)).values[:, -1:]
                    lg = lg.masked_fill(lg < kth, float("-inf"))
                if top_p < 1.0:
                    sl, si = torch.sort(lg, descending=True, dim=-1)
                    pr = torch.softmax(sl, dim=-1)
                    drop = (pr.cumsum(-1) - pr) > top_p                # keep the token that crosses top_p
                    sl = sl.masked_fill(drop, float("-inf"))
                    lg = torch.full_like(lg, float("-inf")).scatter(1, si, sl)
                nxt = torch.multinomial(torch.softmax(lg, dim=-1), 1).squeeze(1)

            nxt = torch.where(finished, torch.full_like(nxt, pad_id), nxt)
            for b in range(B):
                if not finished[b] and nxt[b].item() != eos_id:
                    out[b].append(int(nxt[b].item()))
            finished = finished | (nxt == eos_id)
            seen.scatter_(1, nxt[:, None], True)
            if bool(finished.all()):
                break
            key_valid = torch.cat([key_valid, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)
            logits, cache = model(nxt[:, None], use_cache=True, cache=cache,
                                  start_pos=L + step, key_valid=key_valid)
    return out

Exact unchanged source copy from the Phase 2 v2 cell beginning `%%writefile bpe_tokenizer.py`. Its matching v2 tokenizer repository is configured above.

In [ ]:
%%writefile bpe_tokenizer.py
#!/usr/bin/env python3
import os, json, regex
from typing import Dict, List, Tuple
from collections import Counter, defaultdict
from tqdm.auto import tqdm

import regex


_PAT = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")


class PureByteLevelBPETokenizer:
    def __init__(self, vocab_size: int = 33000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]

        self.encoder: Dict[bytes, int] = {}
        self.decoder: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], bytes] = {}       # pair -> merged token
        self.merge_ranks: Dict[Tuple[bytes, bytes], int] = {}    # pair -> order learned (lower = earlier)
        self._cache: Dict[str, List[int]] = {}

        self._init_base_vocab()

    def _init_base_vocab(self):
        for b in range(256):
            b_bytes = bytes([b])
            self.encoder[b_bytes] = b
            self.decoder[b] = b_bytes

        # special tokens
        for idx, token in enumerate(self.special_tokens, start=256):
            b_bytes = token.encode("utf-8")
            self.encoder[b_bytes] = idx
            self.decoder[idx] = b_bytes

    @property
    def pad_token_id(self) -> int:
        return self.encoder["<pad>".encode("utf-8")]

    @property
    def unk_token_id(self) -> int:
        return self.encoder["<unk>".encode("utf-8")]

    @property
    def bos_token_id(self) -> int:
        return self.encoder["<s>".encode("utf-8")]

    @property
    def eos_token_id(self) -> int:
        return self.encoder["</s>".encode("utf-8")]

    # ------------------------------------------------------------------ training
    @staticmethod
    def _merge_word(word, a, b, new):
        out, i, n = [], 0, len(word)
        while i < n:
            if i < n - 1 and word[i] == a and word[i + 1] == b:
                out.append(new)
                i += 2
            else:
                out.append(word[i])
                i += 1
        return out

    def train_from_texts(self, texts: List[str], target_vocab_size: int = None, min_word_freq: int = 2):
        if target_vocab_size is None:
            target_vocab_size = self.vocab_size
        num_merges = target_vocab_size - len(self.encoder)
        print(f"\nTraining byte level BPE from scratch. Target merges: {num_merges:,} | Base Vocab: {len(self.encoder)}")

        # 1) pre-tokenize + count unique words (the BPE then runs on this small table, not the raw text)
        word_counts = Counter()
        for text in tqdm(texts, desc="Counting pre-tokens"):
            if text and text.strip():
                word_counts.update(_PAT.findall(text))

        words, freqs = [], []
        for w, c in word_counts.items():
            if c >= min_word_freq:
                words.append([bytes([b]) for b in w.encode("utf-8", "replace")])
                freqs.append(c)
        del word_counts
        print(f"Unique words used for merge learning: {len(words):,}")

        # 2) initial pair statistics (updated incrementally afterwards)
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for i, w in enumerate(tqdm(words, desc="Initial pair counts")):
            for p in zip(w, w[1:]):
                pair_counts[p] += freqs[i]
                where[p].add(i)

        # 3) merge loop
        pbar = tqdm(total=num_merges, desc="Learning BPE Merges", dynamic_ncols=True)
        for _ in range(num_merges):
            if not pair_counts:
                print("No more candidate pairs found to merge.")
                break

            best_pair, freq = max(pair_counts.items(), key=lambda kv: kv[1])
            if freq < 2:
                print(f"Stopping early: Most frequent pair frequency is {freq}")
                break

            a, b = best_pair
            merged_token = a + b
            self.merges[best_pair] = merged_token
            self.merge_ranks[best_pair] = len(self.merge_ranks)
            if merged_token not in self.encoder:          # two merge paths can give the same bytes
                new_id = len(self.encoder)
                self.encoder[merged_token] = new_id
                self.decoder[new_id] = merged_token

            for i in list(where[best_pair]):
                w, f = words[i], freqs[i]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        del pair_counts[p]
                w = self._merge_word(w, a, b, merged_token)
                words[i] = w
                for p in zip(w, w[1:]):
                    pair_counts[p] += f
                    where[p].add(i)
            del where[best_pair]
            pair_counts.pop(best_pair, None)

            pbar.update(1)
            pbar.set_postfix({"vocab_size": len(self.encoder), "top_pair_freq": freq})

        pbar.close()
        self._cache.clear()
        print(f"Training finished! Final Vocab Size: {len(self.encoder):,}")

    # ------------------------------------------------------------------ encode / decode
    def _encode_word(self, word: str) -> List[int]:
        cached = self._cache.get(word)
        if cached is not None:
            return cached
        tokens = [bytes([b]) for b in word.encode("utf-8", "replace")]
        while len(tokens) >= 2:
            best_pair, best_rank = None, None
            for p in zip(tokens, tokens[1:]):
                r = self.merge_ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_pair, best_rank = p, r
            if best_pair is None:
                break
            tokens = self._merge_word(tokens, best_pair[0], best_pair[1], self.merges[best_pair])
        ids = [self.encoder.get(t, self.unk_token_id) for t in tokens]
        if len(self._cache) < 1_000_000:
            self._cache[word] = ids
        return ids

    def encode(self, text: str) -> List[int]:
        ids: List[int] = []
        for word in _PAT.findall(text):
            ids.extend(self._encode_word(word))
        return ids

    def decode(self, token_ids: List[int]) -> str:
        raw_bytes = b"".join(self.decoder.get(i, b"<unk>") for i in token_ids)
        return raw_bytes.decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ save / load
    def save(self, save_dir: str):
        os.makedirs(save_dir, exist_ok=True)
        vocab_path = os.path.join(save_dir, "vocab.json")
        merges_path = os.path.join(save_dir, "merges.json")
        config_path = os.path.join(save_dir, "tokenizer_config.json")

        # latin1 maps bytes 1:1 onto code points, so any byte string survives JSON
        ser_vocab = {k.decode("latin1"): v for k, v in self.encoder.items()}
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(ser_vocab, f, ensure_ascii=False, indent=2)

        # JSON (not a space-separated text file): tokens can contain spaces, newlines, \r, \xa0 ...
        ser_merges = [[p1.decode("latin1"), p2.decode("latin1")] for (p1, p2) in self.merges]  # in learned order
        with open(merges_path, "w", encoding="utf-8") as f:
            json.dump(ser_merges, f, ensure_ascii=False)

        config = {
            "tokenizer_class": "PureByteLevelBPETokenizer",
            "vocab_size": len(self.encoder),
            "special_tokens": self.special_tokens,
            "pad_token_id": self.pad_token_id,
            "bos_token_id": self.bos_token_id,
            "eos_token_id": self.eos_token_id
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

        print(f"Tokenizer saved successfully to {save_dir}")

    @classmethod
    def load(cls, save_dir: str) -> "PureByteLevelBPETokenizer":
        tok = cls()
        with open(os.path.join(save_dir, "vocab.json"), "r", encoding="utf-8") as f:
            ser_vocab = json.load(f)
        tok.encoder = {k.encode("latin1"): v for k, v in ser_vocab.items()}
        tok.decoder = {v: k.encode("latin1") for k, v in ser_vocab.items()}

        tok.merges, tok.merge_ranks = {}, {}
        with open(os.path.join(save_dir, "merges.json"), "r", encoding="utf-8") as f:
            for rank, (s1, s2) in enumerate(json.load(f)):
                p1, p2 = s1.encode("latin1"), s2.encode("latin1")
                tok.merges[(p1, p2)] = p1 + p2
                tok.merge_ranks[(p1, p2)] = rank
        return tok


Shared implementation: LoRMA+, equivalent fused GQA attention, majority-coverage selection, aligned reference targets, cached generation and generated-sentence reduction.

In [ ]:
%%writefile phase6_v2_common.py
import os, json, math, random, re, time, hashlib
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import asdict
from llama_scratch import Config, LlamaLM
from bpe_tokenizer import PureByteLevelBPETokenizer

def read_config():
    return json.loads(Path('phase6_v2_config.json').read_text())

def seed_all(seed):
    random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def atomic_json(obj, path):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf8'); tmp.replace(path)

class LoRMALinear(nn.Module):
    # Identity-inflated LoRMA+: W_eff=(I + scale*B@A)@W. A/B act on W*x, not on x.
    def __init__(self,base,rank=16,alpha=16,dropout=.05):
        super().__init__();self.base=base;self.scale=alpha/rank
        self.A=nn.Parameter(torch.empty(rank,base.out_features));self.B=nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5));self.drop=nn.Dropout(dropout)
    def forward(self,x):
        y=self.base(x)
        return y+F.linear(F.linear(self.drop(y),self.A),self.B)*self.scale

def inject_lorma(model,c):
    model.requires_grad_(False)
    for block in model.layers:
        for parent,names in [(block.attn,['wq','wk','wv','wo']),(block.mlp,['w_gate','w_up','w_down'])]:
            for name in names:setattr(parent,name,LoRMALinear(getattr(parent,name),c['rank'],c['alpha'],c['dropout']))
    for name,p in model.named_parameters():
        if 'norm' in name:p.requires_grad_(True)
    return model

def merge_lorma(model):
    for parent in list(model.modules()):
        for name,m in list(parent.named_children()):
            if isinstance(m,LoRMALinear):
                with torch.no_grad():m.base.weight.add_((m.B@(m.A@m.base.weight))*m.scale)
                setattr(parent,name,m.base)
    return model

def load_base(c):
    ck=torch.load(c['checkpoint'],map_location='cpu',weights_only=False)
    assert isinstance(ck,dict) and 'cfg' in ck and 'model' in ck,'Use the original v2 LM checkpoint with cfg/model, not NER weights.'
    cfg=Config(**ck['cfg']);cfg.attn_impl='sdpa'
    assert c['input_tokens']+c['target_tokens']<=cfg.max_seq_len
    m=LlamaLM(cfg);m.load_state_dict(ck['model'],strict=True)
    assert all(torch.isfinite(p).all() for p in m.parameters()),'Nonfinite input model'
    return m

def hidden(model,x):
    h=model.tok_emb(x); co=si=None
    if model.cfg.pos_enc=='rope': co,si=model.rope_cos[:x.shape[1]],model.rope_sin[:x.shape[1]]
    # Right padding needs no key mask: causal attention means real tokens cannot see later pads.
    for b in model.layers: h,_=b(h,co,si,model.alibi_slopes,None,None,False)
    return model.norm(h)

class Objective(nn.Module):
    def __init__(self,model):super().__init__();self.model=model
    def forward(self,x,y):
        h=hidden(self.model,x);mask=y.ne(-100);z=self.model.lm_head(h[mask]).float()
        return F.cross_entropy(z,y[mask])

def collate(examples,pad,device):
    length=max(len(e['x']) for e in examples)
    x=torch.full((len(examples),length),pad,dtype=torch.long); y=torch.full_like(x,-100)
    for i,e in enumerate(examples):
        x[i,:len(e['x'])]=torch.tensor(e['x']); y[i,:len(e['y'])]=torch.tensor(e['y'])
    return x.to(device),y.to(device)

def token_ids(tok,text):
    ids=tok.encode(text); return ids.ids if hasattr(ids,'ids') else ids

def make_target(tok,text,limit):
    raw=token_ids(tok,text)
    # A cropped reference is not an actual end of summary: do not teach false early EOS.
    return (raw+[tok.eos_token_id] if len(raw)<limit else raw[:limit]), len(raw)>=limit

def chunk_document(tok,text,c):
    prefix=token_ids(tok,'Legal judgment excerpt:\n')
    suffix=token_ids(tok,'\nSummarize the important facts, legal reasoning and outcome stated in this excerpt only:\n')
    budget=c['input_tokens']-1-len(prefix)-len(suffix)
    overlap=c['chunk_overlap']; assert 0<=overlap<budget and budget>64
    ids=token_ids(tok,text); result=[]; start=0
    while start<len(ids):
        end=min(start+budget,len(ids))
        result.append(dict(start=start,end=end,prompt=[tok.bos_token_id]+prefix+ids[start:end]+suffix,text=tok.decode(ids[start:end])))
        if end==len(ids):break
        start=end-overlap
    assert result and result[0]['start']==0 and result[-1]['end']==len(ids)
    assert all(b['start']<=a['end'] for a,b in zip(result,result[1:]))
    return result

def make_prompt(tok,text,c):
    # Convenience for one excerpt; long-document callers MUST use chunk_document.
    chunks=chunk_document(tok,text,c)
    assert len(chunks)==1, 'Long input: use chunk_document to preserve every source token.'
    return chunks[0]['prompt']

def word_features(text):
    stop=set('the and that this with from were have been which their there court case said was for are not but had has its'.split())
    return set(re.findall(r'[a-z]{3,}',text.lower()))-stop

def interval_coverage(chunks):
    merged=[]
    for ch in sorted(chunks,key=lambda x:x['start']):
        if merged and ch['start']<=merged[-1][1]:merged[-1][1]=max(merged[-1][1],ch['end'])
        else:merged.append([ch['start'],ch['end']])
    return sum(b-a for a,b in merged)

def select_chunks(chunks,fraction):
    assert .5<fraction<=1
    if len(chunks)<=2:return chunks[:]
    total=chunks[-1]['end'];n=len(chunks)
    cues=set('held dismissed allowed appeal liable entitled therefore consequently constitution section evidence jurisdiction order judgment'.split())
    ranked=sorted(range(1,n-1),key=lambda i:(len(word_features(chunks[i].get('text','')) & cues),i/n),reverse=True)
    spread=[];queue=[(1,n-2)]
    while queue:
        lo,hi=queue.pop(0)
        if lo>hi:continue
        mid=(lo+hi)//2;spread.append(mid);queue.extend([(lo,mid-1),(mid+1,hi)])
    priority=[0,n-1]
    for a,b in zip(ranked,spread):priority.extend([a,b])
    chosen=[];seen=set()
    for i in priority:
        if i in seen:continue
        seen.add(i);chosen.append(chunks[i])
        if interval_coverage(chosen)/total>=fraction:break
    selected=sorted(chosen,key=lambda x:x['start'])
    assert interval_coverage(selected)/total>=fraction and selected[0]['start']==0 and selected[-1]['end']==total
    return selected

def aligned_examples(tok,record,c):
    from collections import Counter
    all_chunks=chunk_document(tok,record['document'],c)
    chunks=select_chunks(all_chunks,c['train_coverage'])
    features=[word_features(ch['text']) for ch in chunks]; df=Counter(w for fs in features for w in fs)
    idf={w:1+math.log((len(chunks)+1)/(n+1)) for w,n in df.items()}
    norms=[math.sqrt(sum(idf[w]**2 for w in fs)) or 1 for fs in features]
    assigned=[[] for ch in chunks]; omitted=0
    sentences=[s.strip() for s in re.split(r'(?<=[.!?])\s+|\n+',record['summary']) if s.strip()]
    for sentence in sentences:
        query=word_features(sentence); qnorm=math.sqrt(sum(idf.get(w,1)**2 for w in query)) or 1
        scores=[sum(idf[w]**2 for w in query & fs)/(norm*qnorm) for fs,norm in zip(features,norms)]
        best=max(range(len(scores)),key=scores.__getitem__)
        if scores[best]>=c['alignment_min_similarity']: assigned[best].append(sentence)
        else: omitted+=1
    examples=[]
    for i,(chunk,gold) in enumerate(zip(chunks,assigned)):
        if not gold:continue
        target,truncated=make_target(tok,' '.join(gold),c['target_tokens']); prompt=chunk['prompt']
        examples.append(dict(id=record['id'],chunk_index=i,x=(prompt+target)[:-1],y=[-100]*(len(prompt)-1)+target,truncated=truncated))
    return examples,dict(chunks=len(chunks),reference_sentences=len(sentences),unaligned_reference_sentences=omitted)

def prediction_tasks(inputs,c):
    tasks=[]
    for index,row in enumerate(inputs):
        n=len(row['chunks'])
        cap=max(c['min_chunk_new_tokens'],min(c['max_chunk_new_tokens'],math.ceil(c['summary_tokens']/n)))
        cap=int(math.ceil(cap/16)*16)
        for ci,ch in enumerate(row['chunks']):tasks.append(dict(index=index,chunk_index=ci,prompt=ch['prompt'],max_new=cap))
    return tasks

def task_batches(tasks,c,rank):
    ordered=sorted(tasks,key=lambda t:(t['max_new'],len(t['prompt']),t['index'],t['chunk_index']))
    mine=ordered[rank::2]; batches=[]
    for task in mine:
        if not batches or len(batches[-1])==c['gen_batch'] or batches[-1][0]['max_new']!=task['max_new']:batches.append([])
        batches[-1].append(task)
    return batches

def merge_chunk_summaries(tok,texts,c):
    candidates=[]; features=[]
    for chunk_i,text in enumerate(texts):
        for sentence in re.split(r'(?<=[.!?])\s+|\n+',text):
            sentence=sentence.strip(); fs=word_features(sentence)
            if not fs:continue
            if any(len(fs & prev)/max(1,len(fs | prev))>=.75 for prev in features):continue
            features.append(fs); candidates.append((chunk_i,len(candidates),sentence,fs))
    assert candidates, 'All chunk generations were empty/unusable'
    # Compress generated sentences only; no source extraction masquerades as model output.
    cues=set('held dismissed allowed appeal liable entitled therefore order judgment constitution section'.split())
    priority=[]
    for i,(ci,order,sentence,fs) in enumerate(candidates):
        score=len(fs & cues)*2+math.log1p(len(fs))
        if i==0:score+=4
        if ci==len(texts)-1:score+=4
        priority.append((score,i))
    chosen=[]; room=c['summary_tokens']
    for _,i in sorted(priority,reverse=True):
        ids=token_ids(tok,candidates[i][2]+' ')
        if len(ids)<=room:chosen.append(i);room-=len(ids)
    if not chosen:
        return tok.decode(token_ids(tok,candidates[0][2])[:c['summary_tokens']]).strip()
    return ' '.join(candidates[i][2] for i in sorted(chosen)).strip()

def assemble_predictions(tok,inputs,shards,c):
    mapping={}
    for p in shards:
        key=(p['index'],p['chunk_index']); assert key not in mapping, 'Duplicate chunk prediction'
        mapping[key]=p['summary']
    expected={(i,ci) for i,r in enumerate(inputs) for ci in range(len(r['chunks']))}
    assert set(mapping)==expected,'Missing or unexpected chunk predictions'
    return [dict(index=i,id=r['id'],split=r['split'],summary=merge_chunk_summaries(tok,[mapping[i,ci] for ci in range(len(r['chunks']))],c)) for i,r in enumerate(inputs)]

def records(path,training):
    obj=json.loads(Path(path).read_text(encoding='utf8'))
    if isinstance(obj,dict):
        obj=next((obj[k] for k in ['data','records','examples'] if isinstance(obj.get(k),list)),obj)
    assert isinstance(obj,list), 'Expected list of records'
    out=[]; seen=set(); dropped=[]
    for r in obj:
        assert isinstance(r,dict) and 'id' in r and isinstance(r.get('document'),str) and r['document'].strip()
        key=str(r['id']); assert key not in seen, 'Duplicate id'; seen.add(key)
        if training and not (isinstance(r.get('summary'),str) and r['summary'].strip()): dropped.append(key); continue
        out.append(dict(r,id=key))
    return out,dropped

def install_fast_gqa():
    from llama_scratch import GQAttention,apply_rope,repeat_kv
    def forward(self,x,cos,sin,slopes,key_valid,past,use_cache):
        B,T,_=x.shape;q=self.wq(x).view(B,T,self.nh,self.hd).transpose(1,2)
        k=self.wk(x).view(B,T,self.nkv,self.hd).transpose(1,2);v=self.wv(x).view(B,T,self.nkv,self.hd).transpose(1,2)
        if cos is not None:q,k=apply_rope(q,cos,sin),apply_rope(k,cos,sin)
        if past is not None:k=torch.cat([past[0],k],2);v=torch.cat([past[1],v],2)
        cache=(k,v) if use_cache else None;S=k.shape[2];k=repeat_kv(k,self.n_rep);v=repeat_kv(v,self.n_rep)
        if key_valid is None and T==S and slopes is None:
            out=F.scaled_dot_product_attention(q,k,v,is_causal=True)
        else:
            rel=torch.arange(S-T,S,device=x.device)[:,None]-torch.arange(S,device=x.device)[None,:]
            allowed=(rel>=0)[None,None]
            if key_valid is not None:allowed=allowed & key_valid[:,None,None,:]
            if slopes is None:mask=allowed
            else:
                bias=-slopes.view(1,-1,1,1)*rel.clamp(min=0).float()[None,None]
                mask=bias.masked_fill(~allowed,torch.finfo(q.dtype).min).to(q.dtype)
            out=F.scaled_dot_product_attention(q,k,v,attn_mask=mask)
        return self.wo(out.transpose(1,2).reshape(B,T,self.nh*self.hd)),cache
    GQAttention.forward=forward

@torch.inference_mode()
def decode_batch(model,prompts,tok,max_new,force_length=False):
    model.eval(); dev=next(model.parameters()).device; B=len(prompts); L=max(map(len,prompts))
    x=torch.full((B,L),tok.pad_token_id,dtype=torch.long,device=dev); valid=torch.zeros_like(x,dtype=torch.bool)
    for i,p in enumerate(prompts): x[i,L-len(p):]=torch.tensor(p,device=dev); valid[i,L-len(p):]=True
    done=torch.zeros(B,dtype=torch.bool,device=dev); out=[[] for _ in prompts]; seen=torch.zeros(B,model.cfg.vocab_size,dtype=torch.bool,device=dev)
    with torch.autocast(dev.type,dtype=torch.float16,enabled=dev.type=='cuda'):
        logits,cache=model(x,key_valid=valid,use_cache=True)
        for j in range(max_new):
            z=logits[:,-1].float(); z[:,[tok.pad_token_id,tok.bos_token_id,tok.unk_token_id]]=-float('inf')
            if j<32 or force_length: z[:,tok.eos_token_id]=-float('inf')
            z=torch.where(seen,torch.where(z>0,z/1.1,z*1.1),z)
            # Ban repeated generated trigrams without penalizing words copied from the source.
            for b,seq in enumerate(out):
                if len(seq)>=2:
                    banned=[seq[k+2] for k in range(len(seq)-2) if seq[k:k+2]==seq[-2:]]
                    if banned: z[b,banned]=-float('inf')
            nxt=z.argmax(-1); nxt=torch.where(done,torch.full_like(nxt,tok.pad_token_id),nxt)
            values=nxt.tolist()
            for b,v in enumerate(values):
                if not done[b].item() and v!=tok.eos_token_id: out[b].append(v)
            done|=nxt.eq(tok.eos_token_id); seen.scatter_(1,nxt[:,None],True)
            if bool(done.all()) or j+1==max_new: break
            valid=torch.cat([valid,torch.ones(B,1,dtype=torch.bool,device=dev)],dim=1)
            logits,cache=model(nxt[:,None],key_valid=valid,use_cache=True,cache=cache,start_pos=L+j)
    return [tok.decode(s).strip() for s in out]


Execution engine: preparation, GPU benchmark, majority-coverage policy, two-process training, best adapter state in memory, distributed inference, validation metrics, prediction-only packaging and temporary-data cleanup.

In [ ]:
%%writefile phase6_v2_engine.py
import pickle, contextlib, gc
from phase6_v2_common import *

def dump_pickle(obj,path):
    path=Path(path);tmp=path.with_suffix('.tmp')
    with tmp.open('wb') as f:pickle.dump(obj,f,protocol=pickle.HIGHEST_PROTOCOL)
    tmp.replace(path)

def load_pickle(path):
    with Path(path).open('rb') as f:return pickle.load(f)

def prepare(c):
    from huggingface_hub import hf_hub_download
    token=os.environ.get('HF_TOKEN');runtime=Path(c['runtime']);runtime.mkdir(parents=True,exist_ok=True)
    files=[hf_hub_download(c['dataset_repo'],remote,repo_type='dataset',token=token) for remote in [c['train_remote'],c['test_remote']]]
    for file in ['vocab.json','merges.json','tokenizer_config.json']:hf_hub_download(c['tokenizer_repo'],file,local_dir=c['tokenizer_dir'],token=token)
    tok=PureByteLevelBPETokenizer.load(c['tokenizer_dir']);base=load_base(c)
    assert tok.vocab_size==base.cfg.vocab_size
    assert [tok.pad_token_id,tok.unk_token_id,tok.bos_token_id,tok.eos_token_id]==[256,257,258,259]
    train,dropped=records(files[0],True);test,_=records(files[1],False);assert len(train)>2 and test
    random.Random(c['seed']).shuffle(train);nv=max(2,round(len(train)*.05));validation=train[:nv];train=train[nv:]
    examples=[];audit=[]
    for i,row in enumerate(train):
        ex,stats=aligned_examples(tok,row,c);examples.extend(ex);audit.append(stats)
        if (i+1)%500==0:print('Prepared',i+1,'documents;',len(examples),'aligned examples',flush=True)
    val=[]
    for row in validation[:c['val_loss_docs']]:val.extend(aligned_examples(tok,row,c)[0])
    random.Random(c['seed']).shuffle(val);val=val[:c['val_loss_chunks']]
    assert len(examples)>=64 and val,'Too few aligned examples: inspect summary schema and alignment threshold.'
    inputs=[]
    for split,rows in [('validation',validation[:c['val_gen_docs']]),('test',test)]:
        for row in rows:inputs.append(dict(id=row['id'],split=split,chunks=chunk_document(tok,row['document'],c),reference=row.get('summary')))
    # Check the genuine checkpoint/tokenizer together before training.
    dev=torch.device('cuda:0');base=base.to(dev).eval();ids=token_ids(tok,train[0]['document'])[:256]
    with torch.no_grad(),torch.autocast('cuda',dtype=torch.float16):sanity=float(base(torch.tensor([ids[:-1]],device=dev),targets=torch.tensor([ids[1:]],device=dev)))
    assert math.isfinite(sanity) and sanity<8,f'LM sanity failed ({sanity}); verify the exact v2 tokenizer and checkpoint.'
    dump_pickle(dict(train=examples,val=val,inputs=inputs),runtime/'prepared.pkl')
    print(json.dumps(dict(base_config=asdict(base.cfg),lm_sanity_loss=sanity,train_documents=len(train),train_chunk_examples=len(examples),validation_chunk_examples=len(val),test_documents=len(test),dropped_empty_summary_ids=dropped,training_source_coverage=c['train_coverage'],target_truncation=sum(e['truncated'] for e in examples)/len(examples),reference_sentences=sum(a['reference_sentences'] for a in audit),unaligned_reference_sentences=sum(a['unaligned_reference_sentences'] for a in audit)),indent=2),flush=True)

def synchronized_time(device):
    import torch.distributed as dist
    t=torch.tensor(time.time(),device=device,dtype=torch.float64);dist.all_reduce(t,op=dist.ReduceOp.MAX);return t.item()

def eval_loss(obj,examples,c,device,rank):
    import torch.distributed as dist
    obj.eval();sums=torch.zeros(2,device=device,dtype=torch.float64)
    with torch.no_grad():
        for i in range(rank,len(examples),2):
            x,y=collate([examples[i]],c['pad'],device)
            with torch.autocast('cuda',dtype=torch.float16):ce=obj(x,y)
            tokens=y.ne(-100).sum();sums[0]+=ce.double()*tokens;sums[1]+=tokens
    dist.all_reduce(sums);value=float(sums[0]/sums[1]);assert math.isfinite(value)
    return value

def generate_shard(model,rows,tok,c,rank,label):
    result=[];batches=task_batches(prediction_tasks(rows,c),c,rank)
    for i,batch in enumerate(batches):
        assert time.time()<c['started_at']+c['budget_minutes']*60-20,'Prediction budget exhausted; no incomplete submission is finalized.'
        texts=decode_batch(model,[t['prompt'] for t in batch],tok,batch[0]['max_new'])
        for task,text in zip(batch,texts):
            assert text.strip(),f'Empty summary for {rows[task["index"]]["id"]}'
            result.append(dict(index=task['index'],chunk_index=task['chunk_index'],summary=text))
        atomic_json(result,Path(c['runtime'])/f'{label}_rank{rank}.json')
        if i%10==0 or i+1==len(batches):print(f'{label} GPU{rank}: {i+1}/{len(batches)} batches',flush=True)
    return result

def distributed_run(c):
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
    rank=int(os.environ['LOCAL_RANK']);torch.cuda.set_device(rank);device=torch.device('cuda',rank)
    dist.init_process_group('nccl');assert dist.get_world_size()==2,'Choose Kaggle T4 x2'
    seed_all(c['seed']);torch.set_num_threads(2);install_fast_gqa()
    tok=PureByteLevelBPETokenizer.load(c['tokenizer_dir']);c['pad']=tok.pad_token_id
    data=load_pickle(Path(c['runtime'])/'prepared.pkl');model=load_base(c).to(device).eval()
    prompts=[ch['prompt'] for r in data['inputs'] for ch in r['chunks']];prompts=sorted(prompts,key=len,reverse=True)[:c['gen_batch']]
    torch.cuda.synchronize();start=time.time();decode_batch(model,prompts,tok,64,force_length=True);torch.cuda.synchronize()
    bench=torch.tensor(time.time()-start,device=device);dist.all_reduce(bench,op=dist.ReduceOp.MAX)
    remaining=c['started_at']+c['budget_minutes']*60-synchronized_time(device)
    decision=[None]
    if rank==0:
        for fraction in c['inference_coverage_options']:
            rows=[dict(r,chunks=select_chunks(r['chunks'],fraction)) for r in data['inputs']]
            work=max(sum(batch[0]['max_new']/64 for batch in task_batches(prediction_tasks(rows,c),c,r)) for r in [0,1])
            # One trained-model full run + held-out base generation, not three models as in v3.
            reserve=max(6*60,float(bench)*work*1.5+150)
            if remaining-reserve>=c['minimum_training_minutes']*60:decision[0]=dict(fraction=fraction,reserve=reserve);break
    dist.broadcast_object_list(decision,src=0);assert decision[0] is not None,'Even 55% source coverage cannot fit the one-hour budget on this hardware. No documents are silently omitted.'
    fraction=decision[0]['fraction'];reserve=decision[0]['reserve']
    rows=[dict(r,chunks=select_chunks(r['chunks'],fraction)) for r in data['inputs']]
    coverages=[interval_coverage(r['chunks'])/original['chunks'][-1]['end'] for r,original in zip(rows,data['inputs'])]
    assert all(x>=fraction for x in coverages)
    if rank==0:
        dump_pickle(rows,Path(c['runtime'])/'selected_inputs.pkl')
        print(f'GPU benchmark={float(bench):.2f}s/64-token batch; selected coverage={fraction:.0%}, measured min/mean={min(coverages):.1%}/{sum(coverages)/len(coverages):.1%}; generation reserve={reserve/60:.1f}m',flush=True)
    valrows=[r for r in rows if r['split']=='validation']
    generate_shard(model,valrows,tok,c,rank,'baseline');dist.barrier()
    deadline=c['started_at']+c['budget_minutes']*60-reserve
    # Best model is held in CPU memory as trainable tensors only; no weight checkpoint is written.
    inject_lorma(model,c);model=model.to(device);obj=Objective(model);ddp=DDP(obj,device_ids=[rank],broadcast_buffers=False)
    params=[p for p in model.parameters() if p.requires_grad]
    names={name for name,p in model.named_parameters() if p.requires_grad}
    opt=torch.optim.AdamW(params,lr=c['lr']/c['warmup_steps'],weight_decay=.01)
    scaler=torch.amp.GradScaler('cuda',init_scale=1024)
    baseline=eval_loss(obj,data['val'],c,device,rank);best=float('inf');best_state=None;best_step=0
    rng=random.Random(c['seed']);order=[];pointer=0;successful=attempted=skipped=0;start=time.time()
    probe=next(p for name,p in model.named_parameters() if name.endswith('.B'));original_probe=probe.detach().clone()
    bs=c['micro_batch'];accum=c['grad_accum'];global_batch=2*bs*accum
    if rank==0:print('LoRMA+ baseline validation CE:',baseline,'trainable parameters:',sum(p.numel() for p in params),flush=True)
    def consider_checkpoint():
        nonlocal best,best_state,best_step
        value=eval_loss(obj,data['val'],c,device,rank)
        if value<best:
            best=value;best_step=successful;best_state={name:p.detach().cpu().clone() for name,p in model.named_parameters() if name in names}
        if rank==0:print(f'updates={successful} validation_CE={value:.4f} best={best:.4f} baseline={baseline:.4f}',flush=True)
    while successful<c['max_steps']:
        if synchronized_time(device)>deadline-45:break
        if pointer+global_batch>len(order):
            order=list(range(len(data['train'])));rng.shuffle(order);pointer=0
            order+=(order*((global_batch+len(order)-1)//len(order)))[:(-len(order))%global_batch]
        ddp.train();opt.zero_grad(set_to_none=True);ce_sum=0.
        global_ids=order[pointer:pointer+global_batch]
        global_tokens=sum(sum(t!=-100 for t in data['train'][i]['y']) for i in global_ids)
        for micro in range(accum):
            at=pointer+micro*2*bs+rank*bs;examples=[data['train'][i] for i in order[at:at+bs]]
            x,y=collate(examples,c['pad'],device)
            with (ddp.no_sync() if micro+1<accum else contextlib.nullcontext()):
                with torch.autocast('cuda',dtype=torch.float16):
                    ce=ddp(x,y);loss=ce*y.ne(-100).sum()*2/global_tokens
                finite=torch.tensor(int(torch.isfinite(loss)),device=device);dist.all_reduce(finite,op=dist.ReduceOp.MIN)
                assert finite.item(),'Nonfinite loss: stopped before corrupting weights'
                scaler.scale(loss).backward()
            ce_sum+=float(ce.detach())/accum
        pointer+=global_batch;attempted+=1;scaler.unscale_(opt);norm=torch.nn.utils.clip_grad_norm_(params,1.)
        before=scaler.get_scale();scaler.step(opt);scaler.update();did=int(scaler.get_scale()>=before)
        flags=torch.tensor([did,did],device=device);dist.all_reduce(flags[:1],op=dist.ReduceOp.MIN);dist.all_reduce(flags[1:],op=dist.ReduceOp.MAX)
        assert flags[0]==flags[1],'Ranks disagreed on optimizer update'
        if did:
            successful+=1
            progress=min(1.,max(successful/c['max_steps'],(time.time()-start)/max(1,deadline-start-45)))
            lr=c['lr']*min(1.,(successful+1)/c['warmup_steps'])*max(.1,.5*(1+math.cos(math.pi*progress)))
            for group in opt.param_groups:group['lr']=lr
            if successful==1:assert float((probe-original_probe).abs().max())>0,'Optimizer did not move LoRMA B'
        else:
            skipped+=1;assert skipped<20,'Repeated AMP overflows'
        if rank==0 and (attempted%10==0 or attempted==1):print(f'LoRMA updates={successful} attempted={attempted} AMP_skips={skipped} CE={ce_sum:.4f} grad={float(norm):.3f} elapsed={(time.time()-start)/60:.1f}m',flush=True)
        if did and successful%c['eval_every']==0:consider_checkpoint()
    assert successful>=c['min_updates'],f'Only {successful} successful updates; insufficient valid training time'
    consider_checkpoint();assert best_state is not None
    model.load_state_dict(best_state,strict=False)
    movement=float((probe-original_probe).abs().max());assert movement>0
    if rank==0:
        atomic_json(dict(baseline_validation_ce=baseline,best_validation_ce=best,successful_updates=successful,best_step=best_step,attempted_updates=attempted,amp_skips=skipped,monitored_weight_change=movement,selected_source_coverage=fraction,minimum_measured_coverage=min(coverages),mean_measured_coverage=sum(coverages)/len(coverages)),Path(c['runtime'])/'training_metrics.json')
    del ddp,obj,opt,scaler,best_state,data;gc.collect();torch.cuda.empty_cache()
    merge_lorma(model);model.eval();generate_shard(model,rows,tok,c,rank,'lorma');dist.barrier();dist.destroy_process_group()

def finalize(c):
    import zipfile,shutil,tempfile
    from rouge_score import rouge_scorer
    runtime=Path(c['runtime']);rows=load_pickle(runtime/'selected_inputs.pkl');tok=PureByteLevelBPETokenizer.load(c['tokenizer_dir'])
    shards=[];base=[]
    for rank in [0,1]:
        shards+=json.loads((runtime/f'lorma_rank{rank}.json').read_text(encoding='utf8'))
        base+=json.loads((runtime/f'baseline_rank{rank}.json').read_text(encoding='utf8'))
    predictions=assemble_predictions(tok,rows,shards,c);valrows=[r for r in rows if r['split']=='validation'];baseline=assemble_predictions(tok,valrows,base,c)
    test=[dict(id=p['id'],summary=p['summary']) for p in predictions if p['split']=='test']
    expected=[r['id'] for r in rows if r['split']=='test'];assert [p['id'] for p in test]==expected and len(set(expected))==len(expected)
    assert all(isinstance(p['id'],str) and p['summary'].strip() for p in test)
    scorer=rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'],use_stemmer=True)
    def score(pairs):
        values=[scorer.score(a,b) for a,b in pairs];assert values
        return {key:sum(v[key].fmeasure for v in values)/len(values) for key in ['rouge1','rouge2','rougeL']}
    metrics=dict(untuned_base=score([(r['reference'],p['summary']) for r,p in zip(valrows,baseline)]),lorma=score([(r['reference'],p['summary']) for r,p in zip([r for r in rows if r['split']=='validation'],[p for p in predictions if p['split']=='validation'])]))
    print('SAME-DOCUMENT GENERATION COMPARISON:',json.dumps(metrics,indent=2),flush=True)
    if metrics['lorma']['rougeL']<=metrics['untuned_base']['rougeL']:print('QUALITY CHECK: held-out ROUGE-L did not improve. Do not claim better quality.',flush=True)
    print('TRAINING/COVERAGE:',(runtime/'training_metrics.json').read_text(),flush=True)
    out=Path(c['submission_dir']);folder=out/'submission_lorma';folder.mkdir(parents=True,exist_ok=True)
    atomic_json(test,folder/'prediction.json')
    with zipfile.ZipFile(out/'submission_lorma.zip','w',zipfile.ZIP_DEFLATED) as z:z.write(folder/'prediction.json','prediction.json')
    with zipfile.ZipFile(out/'submission_lorma.zip') as z:assert z.namelist()==['prediction.json'] and json.loads(z.read('prediction.json'))==test
    resolved=runtime.resolve();expected_runtime=Path(tempfile.gettempdir()).resolve()/'phase6_v2_lorma_runtime'
    assert resolved==expected_runtime and resolved.parent==Path(tempfile.gettempdir()).resolve(),'Unsafe cleanup path'
    assert not Path(c['checkpoint']).resolve().is_relative_to(resolved)
    shutil.rmtree(resolved);assert not resolved.exists()
    print('All test predictions saved. No model weights written or exported. Elapsed minutes:',(time.time()-c['started_at'])/60,flush=True)

if __name__=='__main__':
    import sys
    c=read_config();seed_all(c['seed'])
    {'prepare':prepare,'distributed':distributed_run,'finalize':finalize}[sys.argv[1]](c)


In [ ]:
%%writefile phase6_v2_upload.py
# Prediction-only Hugging Face upload, executed inside the notebook's bounded subprocess runner.
import os,json,time,hashlib,tempfile
from pathlib import Path

EXPECTED_FILES=['submission_lorma/prediction.json','submission_lorma.zip']

def config():return json.loads(Path('phase6_v2_config.json').read_text())

def credentials():
    token=os.environ.get('HF_WRITE_TOKEN')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token=UserSecretsClient().get_secret('HF_WRITE')
        except Exception:token=os.environ.get('HF_TOKEN')
    if not token:raise RuntimeError('Enable Kaggle secret HF_WRITE with a Hugging Face token that can write to the configured repo, or set HF_WRITE_TOKEN.')
    return token

def retry(action,tries=3):
    for attempt in range(tries):
        try:return action()
        except Exception as error:
            status=getattr(getattr(error,'response',None),'status_code',None) or getattr(error,'code',None)
            if status in (400,401,403,404) or attempt+1==tries:raise
            time.sleep(2**(attempt+1))

def preflight(c):
    from huggingface_hub import HfApi
    api=HfApi(token=credentials());retry(lambda:api.whoami())
    retry(lambda:api.create_repo(c['hf_repo_id'],repo_type='model',private=True,exist_ok=True))
    # Older Hub clients have auth_check() but no write= parameter.
    import inspect
    checker=getattr(api,'auth_check',None)
    if checker is not None and 'write' in inspect.signature(checker).parameters:
        retry(lambda:checker(c['hf_repo_id'],repo_type='model',write=True))
    else:
        # Same official /auth-check/write endpoint used by current HfApi clients.
        from urllib.request import Request,urlopen
        url=getattr(api,'endpoint','https://huggingface.co').rstrip('/')+'/api/models/'+c['hf_repo_id']+'/auth-check/write'
        def check_write():
            request=Request(url,headers={'Authorization':'Bearer '+credentials()})
            with urlopen(request,timeout=20) as response:
                assert response.status==200,'Repository write access not confirmed'
        retry(check_write)
    print('HF write access verified for',c['hf_repo_id'],flush=True)
    return api

def upload(c):
    from huggingface_hub import hf_hub_download
    import zipfile
    root=Path(c['submission_dir']);prediction=root/EXPECTED_FILES[0];archive=root/EXPECTED_FILES[1]
    assert all((root/name).is_file() for name in EXPECTED_FILES),'Missing prediction artifacts'
    records=json.loads(prediction.read_text(encoding='utf8'))
    assert isinstance(records,list) and records and all(isinstance(r.get('id'),str) and isinstance(r.get('summary'),str) and r['summary'].strip() for r in records)
    assert len({r['id'] for r in records})==len(records)
    with zipfile.ZipFile(archive) as z:
        assert z.namelist()==['prediction.json']
        assert json.loads(z.read('prediction.json'))==records
    api=preflight(c)
    commit=retry(lambda:api.upload_folder(repo_id=c['hf_repo_id'],repo_type='model',folder_path=str(root),path_in_repo=c['hf_path_in_repo'],allow_patterns=EXPECTED_FILES,commit_message='Phase 6 v2 LoRMA+: verified predictions only'))
    oid=commit.oid;token=credentials()
    # Read back at this exact commit; a successful API return alone is not verification.
    with tempfile.TemporaryDirectory(prefix='phase6_v2_hf_verify_') as cache:
        for name in EXPECTED_FILES:
            remote=c['hf_path_in_repo'].strip('/')+'/'+name
            downloaded=retry(lambda remote=remote:hf_hub_download(repo_id=c['hf_repo_id'],filename=remote,repo_type='model',revision=oid,token=token,local_dir=cache,force_download=True))
            assert hashlib.sha256(Path(downloaded).read_bytes()).digest()==hashlib.sha256((root/name).read_bytes()).digest(),f'Upload hash mismatch: {remote}'
    print('HF upload verified byte-for-byte at commit',oid,flush=True)
    print('https://huggingface.co/'+c['hf_repo_id']+'/tree/'+oid+'/'+c['hf_path_in_repo'].strip('/'),flush=True)

if __name__=='__main__':
    import sys
    c=config()
    {'preflight':preflight,'upload':upload}[sys.argv[1]](c)


In [ ]:
# Verify Hugging Face write permission before the expensive run
run_checked([sys.executable,'-u','phase6_v2_upload.py','preflight'])


In [ ]:
# Prepare aligned training data and verify the real v2 checkpoint/tokenizer
run_checked([sys.executable,'-u','phase6_v2_engine.py','prepare'])


In [ ]:
# Train one LoRMA+ model and generate every test document on BOTH GPUs
run_checked([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nproc_per_node=2','phase6_v2_engine.py','distributed'])


In [ ]:
# Verify complete predictions, package prediction.json and remove temporary data
run_checked([sys.executable,'-u','phase6_v2_engine.py','finalize'])
from IPython.display import display,FileLink
import zipfile
out=Path(CONFIG['submission_dir'])
prediction=out/'submission_lorma'/'prediction.json'
archive=out/'submission_lorma.zip'
with zipfile.ZipFile(archive) as z:
    assert z.namelist()==['prediction.json']
    assert json.loads(z.read('prediction.json'))==json.loads(prediction.read_text(encoding='utf8'))
assert not list(out.rglob('*.pt')), 'No model weights should be exported.'
print(f'Local predictions ready in {(time.time()-RUN_STARTED_AT)/60:.2f} minutes; HF upload follows.')
assert not Path(CONFIG['runtime']).exists(), 'Temporary-data cleanup did not complete.'
display(FileLink(str(prediction)))
display(FileLink(str(archive)))


In [ ]:
# Upload only predictions and verify the exact committed bytes
run_checked([sys.executable,'-u','phase6_v2_upload.py','upload'])
print(f'Complete run including verified HF upload: {(time.time()-RUN_STARTED_AT)/60:.2f} minutes.')
assert time.time()-RUN_STARTED_AT<60*60,'Predictions exist, but this execution exceeded one hour.'


## Reading the run
The run prints selected coverage and minimum/mean **unique source-token coverage** over all inference documents. The first and last source windows are always retained. Selection uses only the judgment, including legal cue terms and windows spread across document positions. It is a heuristic and can omit important text; 55–75% coverage is not proof that every essential fact was retained.

The model is trained on reference sentences weakly aligned to the selected training chunks by lexical IDF cosine similarity. This is not human chunk supervision and does not eliminate hallucinations. Every test document is processed; within a document, selected windows deliberately omit some text to respect the time budget. Generated sentences are deduplicated and selected within the final 640-token budget, then restored to document order. No external RAG, external pretrained weights, distillation, student model or multi-method ablation is used.

Inspect successful updates, AMP skips, weight movement and validation CE, plus base-versus-LoRMA ROUGE on the **same eight held-out documents with the same selection/reducer**. Those are validation scores, not test results. If ROUGE-L fails to improve, a quality warning is printed; the notebook never claims that training alone proves better summaries. Eight validation documents are a noisy runtime compromise. Training is time-limited and may not visit every training example.

Authoring checks: original v2 source copies, RoPE/ALiBi GQA real-token/cached-decode equivalence, identity initialization, genuine LoRMA updates, frozen-base gradients, efficient merge equivalence, fixed-batch loss reduction, target alignment/termination, measured majority selection with endpoint anchors, complete disjoint task shards, missing-chunk rejection, actual finalizer, ZIP contents and safe temporary cleanup were checked on CPU. Synthetic finalizer scores are test fixtures, not real model quality. Genuine checkpoint compatibility, actual Kaggle T4 ×2 runtime and real summarization improvement require the full run here.
